# 5.0 Saturation forecasting dashboard

This notebook implements the interactive visualisation dashboard for the **Geospatial Repletion & Saturation Modelling** project.
It loads the trained GBT forecasting model, reads the aggregated infrastructure summaries, generates saturation forecasts for future time steps, and renders an interactive Leaflet/Folium map showing dynamic city bottlenecks.


In [ ]:
import os
import sys
import math
import logging
from pathlib import Path

import folium
from folium.plugins import TimestampedGeoJson
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import GBTRegressionModel

# Find project root robustly by looking for 'src' folder upward
current = Path.cwd()
while current.name and not (current / "src").exists():
    current = current.parent
PROJECT_ROOT = current
sys.path.insert(0, str(PROJECT_ROOT.resolve()))

# Setup native Hadoop binaries on Windows
from src.step_08_bootstrapping import setup_winutils
setup_winutils(PROJECT_ROOT)

# Initialize local Spark Session with adaptive configuration
active_session = SparkSession.getActiveSession()
if active_session is not None:
    spark = active_session
    logging.info("Reusing active Spark Session.")
else: 
    logging.info("Spawning adaptive distributed Spark Session environment...")
    spark_builder = (
        SparkSession.builder
        .appName("Saturation-Forecasting-Dashboard")
        .config("spark.sql.shuffle.partitions", "10")
    )
    if os.name == 'nt':
        spark_builder = spark_builder.config("spark.driver.host", "127.0.0.1")
    master_url = os.environ.get("SPARK_MASTER")
    if not master_url and not any(env.startswith("SPARK_") for env in os.environ):
        spark_builder = spark_builder.master("local[*]") \
                                     .config("spark.driver.memory", "4g")
    spark = spark_builder.getOrCreate()

print(f"SparkSession started successfully. Spark version: {spark.version}")


## 5.1 Load data and model

We load Vienna's district boundaries and the trained GBT forecasting model from storage.

In [ ]:
import json

# Load districts to centre the map
districts_path = PROJECT_ROOT / "data" / "spatial" / "vienna_districts.geojson"
with open(districts_path, "r", encoding="utf-8") as f:
    districts_geojson = json.load(f)

# Load trained GBT model weights
model_path = PROJECT_ROOT / "models" / "gbt_saturation_forecaster"
gbt_model = GBTRegressionModel.load(str(model_path))
print(f"Loaded GBT Saturation Model from: {model_path}")

## 5.2 Build feature DataFrame for future predictions

We read the aggregate infrastructure count file and run the same feature engineering window transformations to prepare features for prediction.

In [ ]:
data_path = PROJECT_ROOT / "data" / "geospatial_output" / "infrastructure_activity_counts.csv"
df_raw = spark.read.csv(str(data_path), header=True, inferSchema=True)

# Capacity scaling mapping
df_capped = df_raw.withColumn(
    "max_capacity",
    F.when(F.col("infrastructure_type") == "bike_path", 3000.0)
     .when(F.col("infrastructure_type") == "pedestrian_zone", 600.0)
     .otherwise(1000.0)
)

w_seg = Window.partitionBy("infrastructure_label", "infrastructure_type").orderBy("time_window_index")
w_roll = w_seg.rowsBetween(-3, -1)

df_features = df_capped

# Lags
for lag_idx in range(1, 7):
    df_features = df_features.withColumn(f"count_lag_{lag_idx}", F.lag("count", lag_idx).over(w_seg))

# Rolling stats
df_features = df_features.withColumn("count_rolling_mean_3", F.avg("count").over(w_roll))
df_features = df_features.withColumn("count_rolling_std_3", F.stddev("count").over(w_roll))

# Delta
df_features = df_features.withColumn("count_delta", F.col("count") - F.col("count_lag_1"))

# Remove null values created by lag windows
df_clean = df_features.na.drop()

# Encode categories
indexer = StringIndexer(inputCol="infrastructure_type", outputCol="infra_type_idx", handleInvalid="keep")
df_indexed = indexer.fit(df_clean).transform(df_clean)

# Assemble features vector
feature_cols = [
    "infra_type_idx", "count",
    "count_lag_1", "count_lag_2", "count_lag_3",
    "count_lag_4", "count_lag_5", "count_lag_6",
    "count_rolling_mean_3", "count_rolling_std_3", "count_delta"
]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
ml_df = assembler.transform(df_indexed)

print("Feature matrix ready for forecasting.")

## 5.3 Generate saturation predictions

We use our loaded model to forecast future saturation indexes.

> [!NOTE]
> **Memory efficiency justification**
> Collecting the predicted saturation levels for the infrastructure segment time windows yields fewer than 3,000 records. This small payload consumes minimal driver memory (less than 2 MB), making it driver-safe for Folium visualisation.


In [ ]:
predictions = gbt_model.transform(ml_df)

# Extract coordinates/features back to drive for Folium visualisation
# Select necessary fields only to preserve memory
pred_rows = (
    predictions.select(
        "infrastructure_label",
        "infrastructure_type",
        "time_window_index",
        "count",
        "max_capacity",
        "prediction"
    )
    .collect()
)

print(f"Generated forecasts for {len(pred_rows)} segment-windows.")

## 5.4 Build interactive saturation map

We map the forecasted saturation index over time onto Vienna's infrastructure. 
We colour code segments as follows:
- **Green**: Saturation Index < 0.3 (Clear)
- **Yellow**: Saturation Index 0.3 - 0.7 (Moderate load)
- **Red**: Saturation Index >= 0.7 (Overloaded / high risk of bottleneck)

### Figure 20: Saturation forecasting map


In [ ]:
# Load infrastructure segment shapes to match labels
ped_zones_path = PROJECT_ROOT / "data" / "spatial" / "vienna_pedestrian_zones.geojson"
bike_paths_path = PROJECT_ROOT / "data" / "spatial" / "vienna_bike_paths.geojson"

with open(ped_zones_path, "r", encoding="utf-8") as f:
    ped_geojson = json.load(f)
with open(bike_paths_path, "r", encoding="utf-8") as f:
    bike_geojson = json.load(f)

# Index geometry coordinates by segment labels for quick mapping
geoms = {}
for feature in ped_geojson["features"]:
    label = feature["properties"].get("ADRESSE", "ped_zone")
    geoms[("pedestrian_zone", label)] = feature["geometry"]

for feature in bike_geojson["features"]:
    label = feature["properties"].get("GIP_STRNAM", "bike_path")
    geoms[("bike_path", label)] = feature["geometry"]

# Create base Folium Map centered on Vienna
vienna_map = folium.Map(location=[48.2082, 16.3738], zoom_start=13, tiles="CartoDB dark_matter")

# Compile Timestamped GeoJSON features representing predicted changes over time
time_features = []
base_time = 1700000000000  # BASE_TIMESTAMP_MS

for row in pred_rows:
    infra_type = row.infrastructure_type
    label = row.infrastructure_label
    win_idx = row.time_window_index
    count = row["count"]
    cap = row.max_capacity
    pred_sat = max(0.0, min(1.2, float(row.prediction)))  # Clamp saturation between 0 and 1.2

    geom = geoms.get((infra_type, label))
    if geom is None:
        continue

    # Determine segment color based on predicted saturation
    if pred_sat >= 0.7:
        color = "#ff0000"  # Red
    elif pred_sat >= 0.3:
        color = "#ffff00"  # Yellow
    else:
        color = "#00ff00"  # Green

    # Construct ISO8601 time string corresponding to this window
    time_ms = base_time + win_idx * 60 * 1000  # 1-minute windows
    import datetime
    time_str = datetime.datetime.fromtimestamp(time_ms / 1000.0, datetime.UTC).isoformat()

    feat = {
        "type": "Feature",
        "geometry": geom,
        "properties": {
            "time": time_str,
            "style": {
                "color": color,
                "weight": 5 if infra_type == "bike_path" else 2,
                "fillColor": color,
                "fillOpacity": 0.6 if infra_type == "pedestrian_zone" else 0.8
            },
            "popup": f"<b>Segment:</b> {label}<br/><b>Type:</b> {infra_type}<br/><b>Predicted Saturation:</b> {pred_sat:.2%}<br/><b>Forecasted Density:</b> {int(count)}/hr"
        }
    }
    time_features.append(feat)

# Add Slider control overlay
TimestampedGeoJson(
    {
        "type": "FeatureCollection",
        "features": time_features
    },
    period="PT1M",
    add_last_point=True,
    auto_play=False,
    loop=False,
    max_speed=1,
    loop_button=True,
    date_options="YYYY-MM-DD HH:mm",
    time_slider_drag_update=True
).add_to(vienna_map)

# Export map HTML file
output_dir = PROJECT_ROOT / "data" / "geospatial_output"
output_dir.mkdir(parents=True, exist_ok=True)
map_path = output_dir / "saturation_forecast_map.html"
vienna_map.save(str(map_path))
print(f"Successfully generated interactive Saturation Forecast Map at: {map_path}")

**Figure 20**: Interactive saturation forecasting dashboard displaying dynamic city bottlenecks and predicted infrastructure load over time in Vienna.


## 5.5 Concluding Spark Session Teardown

To prevent resource allocation leaks on shared host clusters, we conclude by terminating the driver execution environment explicitly.

In [ ]:
try:
    logging.info("Shutting down Spark Session...")
finally:
    spark.stop()
    logging.info("Spark Session terminated successfully.")


## References

*   **Apache Spark. (n.d.).** *Spark Streaming*. Apache Software Foundation. Retrieved from https://spark.apache.org/streaming/
    *Annotation*: Compute framework powering our Structured Streaming micro-batch pipelines.
*   **Zaharia, M., Xin, R. S., Wendell, P., Das, T., Armbrust, M., Dave, A., Meng, X., Rosen, J., Venkataraman, S., Franklin, M. J., Ghodsi, A., Gonzalez, J., Shenker, S., & Stoica, I. (2016).** Apache Spark: A unified engine for big data processing. *Communications of the ACM*, *59*(11), 56-65. https://doi.org/10.1145/2934664
    *Annotation*: Distributed engine used for windowing aggregations and streaming.
*   **Open Government Data Österreich. (n.d.).** *Data.gv.at* [Data set]. Retrieved from https://www.data.gv.at/home?locale=de
    *Annotation*: Austrian OGD portal providing Vienna Bezirksgrenzen, Fußgängerzonen, and Radwege GeoJSON layers.
*   **Stadt Wien. (2026a).** *Bezirksgrenzen Wien* [Data set]. Open Government Data (OGD) Österreich. https://www.data.gv.at/katalog/dataset/stadt-wien_bezirksgrenzenwien
    *Annotation*: Vienna district polygons under CC BY 4.0 AT license.
*   **Stadt Wien. (2026b).** *Fußgängerzonen Wien* [Data set]. Open Government Data (OGD) Österreich. https://www.data.gv.at/katalog/dataset/stadt-wien_fussgaengerzonenwien
    *Annotation*: Vienna pedestrian zone boundary vectors under CC BY 4.0 AT.
*   **Stadt Wien. (2026c).** *Radwege Wien* [Data set]. Open Government Data (OGD) Österreich. https://www.data.gv.at/katalog/dataset/stadt-wien_radvegewien
    *Annotation*: Vienna cycle path polylines under CC BY 4.0 AT.
*   **Leutenegger, S. T., Lopez, M. A., & Edgington, J. (1997).** STR: A simple and efficient algorithm for R-tree packing. In *Proceedings of the 13th International Conference on Data Engineering* (pp. 497-506). https://doi.org/10.1109/ICDE.1997.582015
    *Annotation*: Packed R-tree packing algorithm powering driver Shapely `STRtree` spatial indices.
